In [4]:
def getGEI(silhouetteSeq):
    silSeq = np.array(silhouetteSeq)
    GEI = np.mean(silSeq, axis=0)
    return GEI


def motion_silhouette_image(sequence_images):
    msi = np.zeros_like(sequence_images[0], dtype=np.float32)
    for i in range(1, len(sequence_images)):
        diff = cv2.absdiff(sequence_images[i], sequence_images[i-1])
        msi += diff
    msi = (msi / np.max(msi) * 255).astype(np.uint8)
    return msi


def gait_flow_image(silhouette_sequence):
    if len(silhouette_sequence) < 2:
        print(f"Warning: Silhouette sequence contains only {len(silhouette_sequence)} images. Returning None.")
        return None

    height, width = silhouette_sequence[0].shape
    flow_accumulator = np.zeros((height, width), dtype=np.float32)

    for i in range(1, len(silhouette_sequence)):
        prev = silhouette_sequence[i-1]
        curr = silhouette_sequence[i]
        
        flow = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        flow_accumulator += magnitude

    if np.max(flow_accumulator) > 0:
        flow_accumulator = (flow_accumulator / np.max(flow_accumulator) * 255).astype(np.uint8)
    else:
        print("Warning: Flow accumulator is empty. Returning None.")
        return None

    return flow_accumulator


def getGEnI(silhouetteSeq):
    cimg = None # Consolidate image
    lenSeq = len(silhouetteSeq) # Length of the sequence (no. of silhouettes)
    
    cimg = np.sum(np.array(silhouetteSeq), axis=0)
    
    cimg = cimg/255 # The values of each img is either 255 or 0, so to binarize /=255
    
    P1 = cimg/lenSeq              # p_1(x,y); the prob. the pixel is 1
    P0 = (lenSeq - cimg)/lenSeq   # p_0(x,y); the prob. the pixel is 0
    
    H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
    
    hmin = np.min(H)
    hmax = np.max(H)
    
    GEnI =  (H - hmin)*255/(hmax-hmin) 
    
    return GEnI


In [5]:
import os
import cv2
import numpy as np
import pandas as pd
import zipfile
from scipy.stats import entropy
from multiprocessing import Pool
from functools import partial
import time

def compute_dewgi(image_sequence):
    """
    Compute the Dynamic Entropy-Weighted Gait Image (DEWGI) from a sequence of silhouette images.
    """
    sequence_array = np.array(image_sequence)
    
    gei = np.mean(sequence_array, axis=0)
    geni = getGEnI(image_sequence)
    geni = (geni - np.min(geni)) / (np.ptp(geni) + 1e-10)
    
    msi = np.mean(np.abs(np.diff(sequence_array, axis=0)), axis=0)
    msi = (msi - np.min(msi)) / (np.ptp(msi) + 1e-10)
    
    dewgi = gei * (1 + geni) * (1 + msi)
    dewgi = (dewgi - np.min(dewgi)) / (np.ptp(dewgi) + 1e-10)
    
    return dewgi

def process_subject(args):
    subject_folder, base_folder, output_folder, camera_angles, counter = args
    start_time = time.time()
    
    subject_path = os.path.join(base_folder, subject_folder, subject_folder)
    if not os.path.isdir(subject_path):
        print(f"No inner subject folder found for: {subject_folder}")
        return [], f"{counter}. Finished processing subject: {subject_folder}", 0
    
    metadata = []
    for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
        nm_folder_path = os.path.join(subject_path, nm_folder)
        if not os.path.exists(nm_folder_path):
            print(f"No {nm_folder} folder found for subject: {subject_folder}")
            continue

        for cam_folder in camera_angles:
            cam_path = os.path.join(nm_folder_path, cam_folder)
            if not os.path.isdir(cam_path):
                continue
            
            image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            if not image_files:
                print(f"No images found in {cam_path}")
                continue
            
            sequence_images = []
            for file in image_files:
                image_path = os.path.join(cam_path, file)
                image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if image is not None and image.size > 0:
                    sequence_images.append(image)
                else:
                    print(f"Failed to load image or empty image: {image_path}")
            
            if sequence_images:
                dewgi_feature = compute_dewgi(sequence_images)
                
                if dewgi_feature is not None and dewgi_feature.size > 0:
                    dewgi_image = ((dewgi_feature - dewgi_feature.min()) / (dewgi_feature.max() - dewgi_feature.min()) * 255).astype(np.uint8)
                    
                    dewgi_filename = f"{subject_folder}_{nm_folder}_{cam_folder}.png"
                    dewgi_path = os.path.join(output_folder, 'dewgi_images', dewgi_filename)
                    cv2.imwrite(dewgi_path, dewgi_image)
                    
                    metadata.append({
                        'image_filename': dewgi_filename,
                        'label': int(subject_folder),
                        'cam_angle': int(cam_folder),
                        'nm_sequence': nm_folder
                    })
                else:
                    print(f"Failed to create valid DEWGI feature for {cam_path}")
    
    processing_time = time.time() - start_time
    return metadata, f"{counter}. Finished processing subject: {subject_folder} (Time: {processing_time:.2f} seconds)", processing_time



def create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=None):
    total_start_time = time.time()
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    dewgi_images_folder = os.path.join(output_folder, 'dewgi_images')
    if not os.path.exists(dewgi_images_folder):
        os.makedirs(dewgi_images_folder)
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    process_args = [(subject, base_folder, output_folder, camera_angles, i) 
                    for i, subject in enumerate(subject_folders, 1)]
    
    with Pool() as pool:
        results = pool.map(process_subject, process_args)
    
    all_metadata = [item for result in results for item in result[0]]
    total_processing_time = 0
    for _, message, processing_time in results:
        print(message)
        total_processing_time += processing_time
    
    if all_metadata:
        df = pd.DataFrame(all_metadata)
        csv_path = os.path.join(output_folder, 'dewgi_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Successfully created {len(df)} DEWGI images for {len(df['label'].unique())} subjects.")
        print(f"Metadata saved to: {csv_path}")
    else:
        print("No DEWGI images were successfully created.")
    
    total_time = time.time() - total_start_time
    print(f"Total processing time for all subjects: {total_processing_time:.2f} seconds")
    print(f"Total execution time including overhead: {total_time:.2f} seconds")

def zip_dataset(output_folder, zip_filename):
    print(f"Creating zip file: {zip_filename}")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)
    print(f"Zip file created: {zip_filename}")

# Main execution
if __name__ == "__main__":
    # Define paths and parameters
    base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
    output_folder = '/kaggle/working/dewgi_dataset'
    zip_filename = '/kaggle/working/dewgi_processed_dataset.zip'
    camera_angles = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']

    # Create the DEWGI dataset
    create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=camera_angles)

    # Zip the dataset
    zip_dataset(output_folder, zip_filename)

    print(f"DEWGI dataset has been created and zipped. You can now download {zip_filename} from the Kaggle output.")

/tmp/ipykernel_30/2158627441.py:52: RuntimeWarning: divide by zero encountered in log2
  H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
/tmp/ipykernel_30/2158627441.py:52: RuntimeWarning: divide by zero encountered in log2
  H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
/tmp/ipykernel_30/2158627441.py:52: RuntimeWarning: invalid value encountered in multiply
  H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
/tmp/ipykernel_30/2158627441.py:52: RuntimeWarning: invalid value encountered in multiply
  H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
/tmp/ipykernel_30/2158627441.py:52: RuntimeWarning: divide by zero encountered in log2
  H = -np.nan_to_num(P0*np.log2(P0)) - np.nan_to_num(P1*np.log2(P1)) # Convert nan to 0 as 0*-inf = nan
/tmp/ipykernel_30/2158627441.py:52: Run

1. Finished processing subject: 057 (Time: 26.29 seconds)
2. Finished processing subject: 086 (Time: 29.05 seconds)
3. Finished processing subject: 121 (Time: 25.38 seconds)
4. Finished processing subject: 061 (Time: 25.19 seconds)
5. Finished processing subject: 048 (Time: 29.46 seconds)
6. Finished processing subject: 053 (Time: 27.38 seconds)
7. Finished processing subject: 051 (Time: 25.73 seconds)
8. Finished processing subject: 095 (Time: 27.88 seconds)
9. Finished processing subject: 018 (Time: 21.96 seconds)
10. Finished processing subject: 044 (Time: 29.03 seconds)
11. Finished processing subject: 016 (Time: 26.29 seconds)
12. Finished processing subject: 007 (Time: 26.08 seconds)
13. Finished processing subject: 009 (Time: 28.71 seconds)
14. Finished processing subject: 012 (Time: 32.75 seconds)
15. Finished processing subject: 029 (Time: 32.06 seconds)
16. Finished processing subject: 025 (Time: 27.81 seconds)
17. Finished processing subject: 078 (Time: 24.20 seconds)
18. Fi